# 04 — Aprendizaje no supervisado (DBSCAN/OPTICS, GMM/KMeans, t-SNE/UMAP)

**Módulo 2 · Algoritmos avanzados de ML (+ MLflow)**

Vemos el **clustering por densidad** (DBSCAN vs OPTICS), el **clustering
probabilístico** (GMM vía EM vs KMeans) y la **reducción de dimensión para
visualización** (t-SNE vs UMAP). Registramos métricas (p. ej. silhouette) y
gráficas como artefactos de **MLflow**.


In [ ]:

import os, sys, warnings
warnings.filterwarnings("ignore")

# Hacemos importable utils/ tanto si el notebook corre desde notebooks/ como
# desde la raíz del repositorio.
_here = os.getcwd()
for cand in (os.path.join(_here, "..", "utils"), os.path.join(_here, "utils"),
             os.path.join(_here, "..", "..", "module2-advanced-ml", "utils")):
    cand = os.path.abspath(cand)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow_helpers import setup_mlflow, log_and_register, register_best_run, registry_available

np.random.seed(42)
print("Versión de MLflow:", mlflow.__version__)


In [ ]:

setup_mlflow("module2-04-unsupervised", backend="dagshub")


## 1. DBSCAN — clustering por densidad

**Intuición:** DBSCAN agrupa puntos que están **densamente empaquetados** y marca
como **ruido** los puntos aislados. Dos hiperparámetros:
- `eps` ($\varepsilon$): radio del vecindario.
- `min_samples` (minPts): puntos necesarios dentro de $\varepsilon$ para ser
  *denso*.

Tipos de punto:
- **Núcleo:** tiene $\ge$ minPts vecinos dentro de $\varepsilon$.
- **Borde:** está dentro de $\varepsilon$ de un núcleo pero no es núcleo.
- **Ruido:** ninguno de los anteriores (etiqueta $-1$).

Los clústeres son conjuntos maximales de puntos *conectados por densidad*. A
diferencia de KMeans, DBSCAN encuentra **formas arbitrarias**, no necesita **$k$**
y es **robusto a outliers** — pero sufre cuando los clústeres tienen **densidades
muy distintas** (un único `eps` global no sirve para todos).


In [ ]:

from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

Xm, _ = make_moons(n_samples=400, noise=0.06, random_state=42)
Xm = StandardScaler().fit_transform(Xm)
db = DBSCAN(eps=0.25, min_samples=5).fit(Xm)
labels = db.labels_
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))

plt.figure(figsize=(6,5))
plt.scatter(Xm[:,0], Xm[:,1], c=labels, cmap="tab10", s=18)
plt.title(f"DBSCAN sobre make_moons: {n_clusters} clústeres, {n_noise} puntos de ruido")
plt.tight_layout(); plt.show()
print("clústeres:", n_clusters, "ruido:", n_noise)


## 2. OPTICS — ordenar puntos para densidad variable

**Intuición:** OPTICS no se compromete con un único `eps`. Calcula para cada punto
una **distancia de alcanzabilidad** y produce un **ordenamiento**. El **gráfico de
alcanzabilidad** muestra valles = clústeres; valles profundos y estrechos son
densos, anchos y poco profundos son esparsos. Puedes extraer clústeres a *distintos*
niveles de densidad — así OPTICS maneja datos de **densidad variable** que DBSCAN
no puede con un solo `eps`.


In [ ]:

from sklearn.cluster import OPTICS
# Tres blobs con densidades muy distintas
Xa, _ = make_blobs(n_samples=[300, 100, 60], centers=[[0,0],[4,4],[8,0]],
                   cluster_std=[0.3, 0.8, 1.6], random_state=42)
opt = OPTICS(min_samples=10, xi=0.05, min_cluster_size=0.05).fit(Xa)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
# Gráfico de alcanzabilidad
space = np.arange(len(Xa))
reach = opt.reachability_[opt.ordering_]
axes[0].plot(space, reach, lw=0.8)
axes[0].set_title("Gráfico de alcanzabilidad de OPTICS (valles = clústeres)")
axes[0].set_xlabel("puntos (orden del clúster)"); axes[0].set_ylabel("dist. alcanzabilidad")
# Clústeres en el espacio de características
axes[1].scatter(Xa[:,0], Xa[:,1], c=opt.labels_, cmap="tab10", s=15)
axes[1].set_title("Clústeres de OPTICS (densidad variable)")
plt.tight_layout(); plt.show()


## 3. KMeans vs modelos de mezcla gaussiana (GMM)

**KMeans** minimiza la distancia cuadrática dentro del clúster:
$$
\min_{\{S_k\}} \sum_{k=1}^{K} \sum_{x\in S_k} \|x-\mu_k\|^2 .
$$
Produce asignaciones **duras**, **esféricas** y de igual varianza.

**GMM** asume que los datos vienen de una mezcla de $K$ gaussianas:
$$
p(x) = \sum_{k=1}^{K} \pi_k\, \mathcal{N}(x \mid \mu_k, \Sigma_k),\qquad \sum_k \pi_k = 1.
$$
Se ajusta por **Esperanza–Maximización (EM)**:

- **Paso E:** calcular responsabilidades suaves (probabilidades posteriores)
$$
\gamma_{ik} = \frac{\pi_k\, \mathcal{N}(x_i\mid\mu_k,\Sigma_k)}{\sum_j \pi_j\, \mathcal{N}(x_i\mid\mu_j,\Sigma_j)} .
$$
- **Paso M:** actualizar los parámetros usando esos pesos
$$
\mu_k = \frac{\sum_i \gamma_{ik} x_i}{\sum_i \gamma_{ik}},\quad
\Sigma_k = \frac{\sum_i \gamma_{ik}(x_i-\mu_k)(x_i-\mu_k)^\top}{\sum_i \gamma_{ik}},\quad
\pi_k = \frac{1}{n}\sum_i \gamma_{ik}.
$$

GMM da asignaciones **suaves** y clústeres **elípticos** (covarianza completa), así
que ajusta clústeres estirados/correlacionados que KMeans falla.


In [ ]:

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# Clústeres anisotrópicos (estirados) para exponer la debilidad de KMeans
Xg, yg = make_blobs(n_samples=600, centers=3, cluster_std=0.7, random_state=42)
transformation = np.array([[0.6, -0.6], [-0.4, 0.8]])
Xg = Xg @ transformation

km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Xg)
gm = GaussianMixture(n_components=3, covariance_type="full", random_state=42).fit(Xg)
km_lab, gm_lab = km.labels_, gm.predict(Xg)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(Xg[:,0], Xg[:,1], c=km_lab, cmap="tab10", s=12)
axes[0].set_title(f"KMeans (silhouette={silhouette_score(Xg, km_lab):.3f})")
axes[1].scatter(Xg[:,0], Xg[:,1], c=gm_lab, cmap="tab10", s=12)
axes[1].set_title(f"GMM (silhouette={silhouette_score(Xg, gm_lab):.3f})")
plt.tight_layout(); plt.show()


In [ ]:

# Registramos métricas de clustering + la comparación como artefactos de MLflow
import tempfile, os
with mlflow.start_run(run_name="kmeans-vs-gmm"):
    mlflow.log_params({"n_clusters": 3, "covariance_type": "full"})
    mlflow.log_metrics({"kmeans_silhouette": silhouette_score(Xg, km_lab),
                        "gmm_silhouette": silhouette_score(Xg, gm_lab),
                        "gmm_bic": gm.bic(Xg), "gmm_aic": gm.aic(Xg)})
    fig, ax = plt.subplots(figsize=(6,4))
    ax.scatter(Xg[:,0], Xg[:,1], c=gm_lab, cmap="tab10", s=12); ax.set_title("Clústeres GMM")
    tmp = os.path.join(tempfile.gettempdir(), "gmm_clusters.png")
    fig.savefig(tmp, dpi=110, bbox_inches="tight"); plt.close(fig)
    mlflow.log_artifact(tmp)
    print("Run de clustering + artefacto registrados:", tmp)


## 4. Reducción de dimensión para visualización: t-SNE vs UMAP

Ambos mapean datos de alta dimensión a 2-D para *visualizar* (no como features).

### t-SNE
Convierte distancias por pares en **probabilidades** (gaussiana en alta-D,
Student-$t$ en baja-D) y minimiza la **divergencia KL** entre ellas:
$$
C = \mathrm{KL}(P \,\|\, Q) = \sum_{i\neq j} p_{ij}\,\log\frac{p_{ij}}{q_{ij}}.
$$
Excelente revelando estructura **local** / clústeres, pero **lento**, estocástico
y las **distancias entre clústeres no son significativas** (mala estructura
global). Knob clave: `perplexity`.

### UMAP
Basado en teoría de variedades / **conjuntos simpliciales difusos**: modela los
datos como un grafo topológico difuso en alta-D y busca un layout en baja-D que lo
iguale. Es **mucho más rápido**, escala mejor y tiende a **preservar mejor la
estructura global** que t-SNE. Knobs clave: `n_neighbors` (local vs global) y
`min_dist` (compacidad de los clústeres).


In [ ]:

from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
digits = load_digits()
Xd, yd = digits.data, digits.target
print("digits:", Xd.shape)

tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
Z_tsne = tsne.fit_transform(Xd)

try:
    import umap
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    Z_umap = reducer.fit_transform(Xd)
    have_umap = True
except Exception as e:
    print("UMAP no disponible (", e, ") — usamos PCA en el panel derecho.")
    from sklearn.decomposition import PCA
    Z_umap = PCA(n_components=2).fit_transform(Xd)
    have_umap = False


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sc = axes[0].scatter(Z_tsne[:,0], Z_tsne[:,1], c=yd, cmap="tab10", s=8)
axes[0].set_title("t-SNE (divergencia KL, estructura local)")
axes[1].scatter(Z_umap[:,0], Z_umap[:,1], c=yd, cmap="tab10", s=8)
axes[1].set_title("UMAP (conjuntos simpliciales difusos)" if have_umap else "PCA (fallback de UMAP)")
fig.colorbar(sc, ax=axes, fraction=0.025, label="dígito")
plt.show()


## 5. Resumen

| Tarea | Método A | Método B | Cuándo gana B |
|---|---|---|---|
| Clustering por densidad | DBSCAN | OPTICS | densidades variables |
| Clustering por centroide | KMeans | GMM | clústeres elípticos / suaves |
| Visualización 2-D | t-SNE | UMAP | velocidad + estructura global |

La calidad del clustering es difícil de medir sin etiquetas — usa **silhouette**,
**BIC/AIC** (GMM) e *inspección del dominio*. Registra todo en MLflow para que las
gráficas y métricas queden adjuntas a cada run.
